# Orchestrating the GPT Training Loop

In our last notebook, we prepared our dataset—the "textbook" for our model. Now, it's time to build the "classroom" and the "teacher" who will guide the model through this textbook. This notebook focuses on the training loop: the engine that drives the learning process, connecting our GPT model with our data and telling it how to improve.

By the end of this notebook, you will be able to implement a complete training loop for a language model from scratch. You will understand the distinct roles of the optimizer, the learning rate scheduler, and the evaluation process in creating a capable model. Finally, you'll learn the practical skill of saving your model's progress so you can resume training later.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import torch.optim as optim
import math
import matplotlib.pyplot as plt

# Set a seed for reproducibility
torch.manual_seed(42)

# ---- Dummy Components for Demonstration ----

# A minimal GPT-like model for demonstration purposes
class SimpleGPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T)) # (T, C)
        x = tok_emb + pos_emb # (B, T, C)
        logits = self.lm_head(x) # (B, T, vocab_size)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

# A dummy data loader that returns random batches
VOCAB_SIZE = 100
BLOCK_SIZE = 8
BATCH_SIZE = 4

def get_batch(split):
    # For this notebook, we'll just generate random data
    # In a real scenario, this would pull from our training or validation data file
    inputs = torch.randint(VOCAB_SIZE, (BATCH_SIZE, BLOCK_SIZE))
    targets = torch.randint(VOCAB_SIZE, (BATCH_SIZE, BLOCK_SIZE))
    return inputs, targets

### The Core Idea: Training as a Feedback Loop

Imagine you're teaching an apprentice chef how to bake bread. You don't just give them the recipe (the model) and a pile of ingredients (the data) and hope for the best. You guide them through a cyclical process of practice and feedback. This is exactly what a training loop does for a neural network.

The process looks like this:

1.  **Attempt the Recipe (Forward Pass):** The apprentice takes a small batch of ingredients (a `batch` of data) and follows the recipe to bake a loaf. The model takes a batch of input tokens and processes them to produce a prediction (logits).

2.  **Taste and Judge (Calculate Loss):** You taste the bread. Is it too salty? Too dense? You compare it to your idea of a perfect loaf. This "difference from perfection" is the `loss`. We calculate how far the model's predictions are from the actual correct answers.

3.  **Pinpoint the Mistake (Backward Pass):** You think back through the recipe steps to figure out *why* the bread is too salty. "Ah, they must have added too much salt at the beginning." This is the `backward pass` or `backpropagation`, where we calculate which parameters in the model were most responsible for the error.

4.  **Give Corrective Feedback (Optimizer Step):** You tell the apprentice, "Next time, use a little less salt." This is the `optimizer step`. The optimizer uses the information from the backward pass to slightly adjust all the model's parameters, nudging them in a direction that should reduce the error.

5.  **Repeat:** The apprentice bakes another loaf with this new instruction, and you repeat the cycle. Over thousands of these loops, their bread gets progressively better.

This entire cycle is the training loop. We just repeat this "attempt, judge, correct" process thousands of times until the model becomes a skilled predictor of the next token.

In [ ]:
# ---- 1. Model and Optimizer Initialization ----
model = SimpleGPT(vocab_size=VOCAB_SIZE, block_size=BLOCK_SIZE, n_embd=32)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

# ---- 2. The Training Loop Itself ----
MAX_STEPS = 100

for step in range(MAX_STEPS):
    # Get a batch of data
    inputs, targets = get_batch('train')

    # Forward pass: let the model attempt the recipe
    logits, loss = model(inputs, targets)

    # Reset gradients from the previous step
    optimizer.zero_grad(set_to_none=True)

    # Backward pass: pinpoint the mistakes
    loss.backward()

    # Optimizer step: give corrective feedback
    optimizer.step()

    if step % 10 == 0:
        print(f"Step {step}, Loss: {loss.item():.4f}")

print(f"Final Loss: {loss.item():.4f}")

Let's walk through that minimal implementation line by line.

**1. Model and Optimizer Initialization**
```python
model = SimpleGPT(vocab_size=VOCAB_SIZE, block_size=BLOCK_SIZE, n_embd=32)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
```
-   First, we create an instance of our `SimpleGPT` model. This is our "apprentice chef."
-   Next, we create an `optimizer`. We're using `AdamW`, a popular and effective choice for training Transformers. We hand it the model's parameters (`model.parameters()`), which are all the numbers (weights and biases) that the optimizer is allowed to tweak. We also give it a `learning rate` (`lr`), which controls how *big* of a correction to make at each step.

**2. The Training Loop**
```python
for step in range(MAX_STEPS):
```
-   This is the main loop that repeats the learning cycle for a fixed number of steps.

```python
    inputs, targets = get_batch('train')
```
-   We fetch a small, manageable chunk of data. `inputs` is what we show the model; `targets` is the correct answer we want it to learn.

```python
    logits, loss = model(inputs, targets)
```
-   This is the **forward pass**. We feed the `inputs` to the model and get back its predictions (`logits`) and a single number, `loss`, which quantifies how wrong the predictions were.

```python
    optimizer.zero_grad(set_to_none=True)
```
-   This is a crucial housekeeping step. PyTorch accumulates gradients from each `loss.backward()` call. We need to reset them to zero before each new backward pass to ensure we are only correcting for the most recent mistake, not all past mistakes combined.

```python
    loss.backward()
```
-   This is the **backward pass**. PyTorch automatically calculates the gradient of the loss with respect to every single parameter in the model. This tells us the direction and magnitude of the change needed for each parameter to reduce the loss.

```python
    optimizer.step()
```
-   This is the **optimizer step**. The optimizer uses the gradients computed in `loss.backward()` to update the model's parameters. This is the moment of learning, where the model is actually improved.

The `print` statement simply lets us monitor the `loss` and see if our model is learning. A decreasing loss is a great sign!

In [ ]:
# ---- Demonstration with Evaluation ----

# Let's create a function to estimate loss on train and validation sets
# This helps us see if the model is truly learning or just memorizing
@torch.no_grad() # Tell PyTorch not to calculate gradients during evaluation
def estimate_loss(model, eval_iters=100):
    out = {}
    model.eval() # Set the model to evaluation mode
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train() # Set the model back to training mode
    return out

# --- Re-initialize model and optimizer ---
model = SimpleGPT(vocab_size=VOCAB_SIZE, block_size=BLOCK_SIZE, n_embd=32)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

# --- Training Loop with Evaluation ---
TRAINING_STEPS = 1000
EVAL_INTERVAL = 100

for step in range(TRAINING_STEPS):
    # Perform one training step
    inputs, targets = get_batch('train')
    logits, loss = model(inputs, targets)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    # Periodically evaluate the model
    if step % EVAL_INTERVAL == 0 or step == TRAINING_STEPS - 1:
        losses = estimate_loss(model)
        print(f"Step {step}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

### Going Deeper: The Learning Rate Scheduler

In our simple loop, the learning rate was fixed at `1e-3`. This is like a teacher giving feedback with the same intensity throughout the entire semester. In the beginning, when the student knows nothing, gentle guidance is helpful. Towards the end, when they are fine-tuning their skills, smaller, more precise corrections are better.

This is the job of a **learning rate scheduler**. It dynamically adjusts the learning rate during training. A very common and effective schedule for Transformers is **cosine decay with warmup**:

1.  **Warmup:** For the first few hundred or thousand steps, the learning rate gradually *increases* from a very small value to its maximum value. Why? In the beginning, the model's weights are random, and a large learning rate could cause it to take a huge, chaotic step in the wrong direction, destabilizing the entire training process. Warmup allows the model to "settle in" before starting to learn at full speed.
2.  **Cosine Decay:** After the warmup phase, the learning rate is slowly decreased, following the shape of a cosine curve, until it reaches a very small value by the end of training. This allows the model to make large progress early on and then make smaller, finer-grained adjustments as it gets closer to a good solution.

This isn't just a minor tweak; using a well-designed learning rate schedule is one of the most critical factors in successfully training large language models. The nanoGPT repository implements exactly this logic.

In [ ]:
# --- Visualizing the Learning Rate Schedule ---

# Constants from nanoGPT's default configuration
LEARNING_RATE = 6e-4
MAX_STEPS = 5000
WARMUP_ITERS = 200
LR_DECAY_ITERS = 5000 # should be ~= max_iters
MIN_LR = 6e-5

lrs = []
for it in range(MAX_STEPS):
    # 1. linear warmup for warmup_iters steps
    if it < WARMUP_ITERS:
        lr = LEARNING_RATE * it / WARMUP_ITERS
    # 2. if it > lr_decay_iters, return min learning rate
    elif it > LR_DECAY_ITERS:
        lr = MIN_LR
    # 3. in between, use cosine decay down to min learning rate
    else:
        decay_ratio = (it - WARMUP_ITERS) / (LR_DECAY_ITERS - WARMUP_ITERS)
        assert 0 <= decay_ratio <= 1
        coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio)) # coeff starts at 1 and goes to 0
        lr = MIN_LR + coeff * (LEARNING_RATE - MIN_LR)
    lrs.append(lr)

# Plot the learning rate schedule
plt.figure(figsize=(10, 5))
plt.plot(lrs)
plt.title("Learning Rate Schedule (Warmup + Cosine Decay)")
plt.xlabel("Training Step")
plt.ylabel("Learning Rate")
plt.grid(True)
plt.show()

# --- Saving and Loading Checkpoints ---
# After training for a while, we want to save our progress.

# Re-initialize for this example
model = SimpleGPT(vocab_size=VOCAB_SIZE, block_size=BLOCK_SIZE, n_embd=32)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

# Imagine we ran the training loop...
# ... and now we want to save.

checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'step': 500, # Let's say we trained for 500 steps
    'loss': 4.1, # And the last loss was 4.1
}

# Save the checkpoint to a file
torch.save(checkpoint, 'model_checkpoint.pt')
print("Checkpoint saved to 'model_checkpoint.pt'")

# --- Now, let's load it back ---
# Create new, untrained instances
new_model = SimpleGPT(vocab_size=VOCAB_SIZE, block_size=BLOCK_SIZE, n_embd=32)
new_optimizer = torch.optim.AdamW(new_model.parameters(), lr=1e-3)

# Load the checkpoint
loaded_checkpoint = torch.load('model_checkpoint.pt')

new_model.load_state_dict(loaded_checkpoint['model_state_dict'])
new_optimizer.load_state_dict(loaded_checkpoint['optimizer_state_dict'])
start_step = loaded_checkpoint['step']
last_loss = loaded_checkpoint['loss']

print(f"Checkpoint loaded. Resuming training from step {start_step} with loss {last_loss:.4f}")
# Now you could resume the training loop using new_model and new_optimizer

### Summary

In this notebook, we constructed the heart of the training process: the training loop. We saw that it's a simple but powerful cycle of getting data, making a prediction, calculating the error, and updating the model.

**What We Built:**
- A complete, functional training loop for a language model.
- An evaluation function (`estimate_loss`) to monitor both training and validation performance, helping us guard against overfitting.
- A practical method for saving and loading model checkpoints, which is essential for any long-running training job.

**Key Takeaways:**
- Training is a feedback loop: **Forward Pass → Loss Calculation → Backward Pass → Optimizer Step**.
- The **optimizer** is the agent that changes the model's weights based on the calculated gradients.
- **Learning rate schedulers** are crucial for stable and efficient training, often involving a "warmup" period followed by a "decay" period.
- Separating training (`model.train()`) and evaluation (`model.eval()`) modes is important because some layers, like Dropout, behave differently in each mode.

**What's Next:**
Our model is now trained! It has learned the patterns and structure of the language from our dataset. In the next notebook, *Generating Text with a GPT Model*, we'll finally get to see the fruits of our labor. We will use our trained model to generate new, original text, completing our journey from raw data to a creative AI.